<a href="https://colab.research.google.com/github/Darklord007masai/part3-churn-model/blob/main/part3_churn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve

# 1. Load Pre-built Modeling Table
rfm_df = pd.read_csv('/content/drive/MyDrive/rfm_modeling_snapshot.csv')

# Handle explicit missing value strategies observed during Part 1
rfm_df['loyalty_tier'] = rfm_df['loyalty_tier'].fillna('Not_Enrolled')

# Define features and target
target_col = 'churn_next_60d'
split_col = 'split'

# Drop tracking columns not utilized as computational modeling features
X_raw = rfm_df.drop(columns=['customer_id', 'snapshot_date', target_col])
y_raw = rfm_df[target_col]
splits = rfm_df[split_col]

# Define feature types
cat_cols = ['city_tier', 'age_group', 'acquisition_channel', 'loyalty_tier', 'preferred_category', 'marketing_consent']
num_cols = [col for col in X_raw.columns if col not in cat_cols and col != 'split']

# 2. Build Pipeline Transformers (Preprocessing)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

# 3. Partition Datasets using exact pre-allocated splits
X_train_raw = X_raw[splits == 'train'].drop(columns=['split'])
y_train = y_raw[splits == 'train']

X_val_raw = X_raw[splits == 'validation'].drop(columns=['split'])
y_val = y_raw[splits == 'validation']

X_test_raw = X_raw[splits == 'test'].drop(columns=['split'])
y_test = y_raw[splits == 'test']

# Transform Datasets
X_train = preprocessor.fit_transform(X_train_raw)
X_val = preprocessor.transform(X_val_raw)
X_test = preprocessor.transform(X_test_raw)

# Extract post-encoding feature names for later importance mappings
cat_encoder = preprocessor.named_transformers_['cat']
encoded_cat_features = list(cat_encoder.get_feature_names_out(cat_cols))
feature_names = np.array(num_cols + encoded_cat_features)

# 4. Train Models
# Baseline: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Champion Model: Optimized Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 5. Evaluate Model Outputs
def get_metrics(model, X, y):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        'Accuracy': accuracy_score(y, preds),
        'Precision': precision_score(y, preds),
        'Recall': recall_score(y, preds),
        'F1-Score': f1_score(y, preds),
        'ROC-AUC': roc_auc_score(y, probs)
    }

print("--- Baseline Logistic Regression (Validation Split) ---")
print(get_metrics(lr_model, X_val, y_val))

print("\n--- Champion Random Forest Classifier (Validation Split) ---")
rf_val_metrics = get_metrics(rf_model, X_val, y_val)
print(rf_val_metrics)

# 6. Generate the Mandatory Performance Visualization Suite
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot A: Confusion Matrix
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0], cbar=False)
axes[0].set_title('Confusion Matrix (Test Split)')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# Subplot B: ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, rf_probs)
axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Area = {roc_auc_score(y_test, rf_probs):.3f}')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend(loc="lower right")

# Subplot C: Feature Importance (Top 10 drivers)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1][:10]

sns.barplot(x=importances[indices], y=feature_names[indices], palette='viridis', ax=axes[2])
axes[2].set_title('Top 10 Feature Importance Drivers')
axes[2].set_xlabel('Relative Importance Score')

plt.tight_layout()
plt.savefig('model_performance_metrics.png', dpi=300)
plt.close()
print("\n[SUCCESS] Verification plot file 'model_performance_metrics.png' successfully written to disk.")

--- Baseline Logistic Regression (Validation Split) ---
{'Accuracy': 0.8154761904761905, 'Precision': 0.8057553956834532, 'Recall': 0.7619047619047619, 'F1-Score': 0.7832167832167832, 'ROC-AUC': np.float64(0.8827340459993521)}

--- Champion Random Forest Classifier (Validation Split) ---
{'Accuracy': 0.8035714285714286, 'Precision': 0.8, 'Recall': 0.7346938775510204, 'F1-Score': 0.7659574468085106, 'ROC-AUC': np.float64(0.8780189324407012)}


/tmp/ipykernel_7551/867724214.py:112: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=importances[indices], y=feature_names[indices], palette='viridis', ax=axes[2])



[SUCCESS] Verification plot file 'model_performance_metrics.png' successfully written to disk.


In [3]:
import joblib

# Define the path to save the model in Google Drive
model_path = '/content/drive/MyDrive/random_forest_churn_model.pkl'

# Save the trained Random Forest model
joblib.dump(rf_model, model_path)

print(f"Random Forest model successfully exported to: {model_path}")

Random Forest model successfully exported to: /content/drive/MyDrive/random_forest_churn_model.pkl
